# 🚀 Gradient Boosting: corregir los errores uno a uno (versión sencilla)
**Curso de IA · Bloque de Machine Learning · Sesión 6 (versión básica)**

---

## La idea en una frase

> Hago una predicción sencilla. Miro **dónde me he equivocado**. Hago otro modelo pequeño **solo para corregir esos errores**. Repito muchas veces. **La suma de todos** es un modelo muy bueno.

## La analogía: el golf ⛳

Un jugador da un primer golpe fuerte que deja la bola **cerca del hoyo, pero no dentro**. Con el segundo golpe corrige la distancia que falta. Con el tercero, lo que todavía falta...

Cada golpe es **pequeño y sencillo**, pero cada uno **corrige el error del anterior**. Al final, la bola entra.

## La receta

```
predicción = media + 0,1 × árbol₁ + 0,1 × árbol₂ + 0,1 × árbol₃ + ...
```

1. Empezamos prediciendo **la media** de todos los datos.
2. Calculamos los **errores** (lo que falta para llegar al valor real). Se llaman **residuos**.
3. Entrenamos un **árbol muy pequeño** para predecir **esos errores**.
4. **Sumamos** su predicción, multiplicada por una cantidad pequeña (la **tasa de aprendizaje**, por ejemplo 0,1), para no pasarnos.
5. Volvemos al paso 2 con la predicción ya mejorada.

> 🌳 Por dentro son **árboles de decisión** muy pequeños (2 o 3 preguntas). Cada árbol es "torpe", pero **la suma de cientos de torpes** hace un modelo muy potente.

## Los pasos de hoy

| Apartado | Paso | En una frase |
|---|------|--------------|
| **A** | **Construir el boosting a mano** | Programarlo nosotros, en 8 líneas |
| 1-2 | Herramientas y datos de juguete | Una curva ondulada |
| 3 | El bucle de boosting | Residuo → árbol → suma |
| 4 | Verlo ronda a ronda | Cómo se aplanan los errores |
| 5 | Comparar con scikit-learn | ¿Hace lo mismo? |
| **B** | **Un caso con 3 variables** | El consumo de un coche |
| 6 | Datos y primer vistazo | Velocidad, peso y temperatura |
| 7 | Comparar modelos | Recta, árbol solo y boosting |
| 8 | Número de árboles y tasa de aprendizaje | ¿Cuánto es "demasiado"? |
| 9 | ¿Qué variables importan? | `feature_importances_` |
| 10 | Resumen | La receta |

## Cómo usar este cuaderno

Ejecuta cada celda con **Shift + Enter**, **en orden**. Si algo se estropea: *Entorno de ejecución → Reiniciar y ejecutar todo*. Antes de ejecutar, **lee y piensa qué va a salir**.

> 👩‍🏫 **Nota para el profesor**
>
> Versión **simplificada** del cuaderno de Gradient Boosting. Se han quitado: la **clasificación** con impagos y sus métricas (AUC, AP), la **parada temprana**, la **dependencia parcial**, la **búsqueda de hiperparámetros**, la importancia por permutación y las variantes (Hist, XGBoost, LightGBM). Se mantiene **lo esencial: el bucle a mano** (que es lo que de verdad hace entender la técnica) y un caso de regresión sencillo.
>
> **Duración orientativa: unas 3 horas** con los ejercicios.

## 1. Preparar las herramientas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Todo listo ✅")

---
# 🅰️ PARTE A — El boosting construido a mano

## 2. Unos datos de juguete

Una sola variable `x` y una **curva ondulada** con un poco de ruido. Queremos predecir `y`.

In [ ]:
rng = np.random.default_rng(3)
x = np.sort(rng.uniform(0, 10, 150))
y = np.sin(x) + 0.3 * x + rng.normal(0, 0.3, 150)
X1 = x.reshape(-1, 1)             # scikit-learn necesita una tabla (150 filas × 1 columna)

plt.scatter(x, y, alpha=0.6, s=18)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Datos con una curva ondulada")
plt.show()

## 3. El bucle de boosting

Seguimos la receta paso a paso. Usamos **árboles muy pequeños** (`max_depth=2`: solo 2 niveles de preguntas) y una **tasa de aprendizaje de 0,3**.

```
prediccion = media de y
repetir 50 veces:
    residuos = y - prediccion              ← lo que falta
    arbol = árbol pequeño que predice los residuos
    prediccion = prediccion + 0,3 × arbol.predict(x)
```

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tasa = 0.3
prediccion = np.full_like(y, y.mean())      # paso 1: empezamos por la media
historia = {0: prediccion.copy()}           # guardamos la predicción de cada ronda

for ronda in range(1, 51):
    residuos = y - prediccion                                                        # paso 2: los errores
    arbol = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X1, residuos)     # paso 3
    prediccion = prediccion + tasa * arbol.predict(X1)                               # paso 4
    historia[ronda] = prediccion.copy()

print("Error medio (al cuadrado) con solo la media:", round(np.mean((y - historia[0]) ** 2), 3))
print("Error medio tras  1 árbol  :", round(np.mean((y - historia[1]) ** 2), 3))
print("Error medio tras 10 árboles:", round(np.mean((y - historia[10]) ** 2), 3))
print("Error medio tras 50 árboles:", round(np.mean((y - historia[50]) ** 2), 3))

👀 **Cada árbol que añadimos reduce el error.** Veámoslo dibujado.

## 4. Verlo ronda a ronda

- **Fila de arriba:** la predicción (línea roja) tras 0, 1, 5 y 50 árboles.
- **Fila de abajo:** los **residuos** (lo que todavía falta por corregir).

In [ ]:
rondas = [0, 1, 5, 50]
fig, ejes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)

for col, r in enumerate(rondas):
    ejes[0, col].scatter(x, y, alpha=0.4, s=10)
    ejes[0, col].plot(x, historia[r], color="red", linewidth=2.5)
    ejes[0, col].set_title("Solo la media" if r == 0 else f"Tras {r} árbol(es)")

    ejes[1, col].scatter(x, y - historia[r], alpha=0.5, s=10, color="gray")
    ejes[1, col].axhline(0, color="black", linewidth=0.8)
    ejes[1, col].set_xlabel("x")

ejes[0, 0].set_ylabel("Predicción")
ejes[1, 0].set_ylabel("Errores que quedan")
plt.tight_layout()
plt.show()

👀 **Esto es lo más importante de la sesión:**

- **Al principio:** la predicción es una línea plana (la media). Los errores tienen **toda la forma de la curva**.
- **Tras 1 árbol:** aparece un primer "escalón". Los errores son menores.
- **Tras 5 árboles:** la línea ya va **siguiendo la curva**.
- **Tras 50 árboles:** la línea roja sigue la curva con suavidad y **los errores son una nube sin forma alrededor de 0**: ya no queda nada por aprender (solo ruido).

Cada árbol es **torpe** (solo 2 niveles), pero **la suma de 50 torpes dibuja una curva suave**. Eso es el boosting.

## 5. Comparar con scikit-learn

scikit-learn tiene este mismo algoritmo ya programado: `GradientBoostingRegressor`. Si lo configuramos igual, ¿obtiene **lo mismo** que nuestro bucle?

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    n_estimators=50,          # 50 árboles (rondas)
    learning_rate=0.3,        # la tasa de aprendizaje
    max_depth=2,              # árboles pequeños
    criterion="squared_error",
    random_state=0,
)
gb.fit(X1, y)

diferencia = np.abs(gb.predict(X1) - historia[50]).max()
print(f"Diferencia máxima entre nuestro bucle y scikit-learn: {diferencia:.6f}")

✅ **Idénticos (diferencia ≈ 0).** No hay magia: `GradientBoostingRegressor` es exactamente el bucle que hemos escrito.

> 💬 **¿Por qué se llama "gradient" (gradiente)?** Porque, para el error al cuadrado, el residuo indica **en qué dirección hay que mover la predicción para reducir el error**, igual que "bajar una cuesta" siguiendo la pendiente. Con eso basta por ahora.

---
# 🅱️ PARTE B — Un caso con tres variables

## 6. Datos: el consumo de un coche

Fabricamos `consumo.csv` con **300 viajes**. Queremos predecir el **consumo** (litros / 100 km) a partir de:

| Columna | Significado |
|---|---|
| `velocidad` | km/h (el consumo forma una **"U"**: alto a velocidad baja y alta) |
| `peso` | Peso del coche con carga (kg) |
| `temperatura` | Temperatura exterior (°C). Con frío, consume más |

> 💡 No hace falta entender el código de esta celda.

In [ ]:
rng = np.random.default_rng(5)
n = 300

velocidad = rng.uniform(20, 140, n).round(0)
peso = rng.normal(1300, 200, n).clip(800, 2000).round(0)
temperatura = rng.normal(15, 8, n).round(1)

consumo = (4.5
           + 0.0011 * (velocidad - 75) ** 2
           + 0.0008 * (peso - 1200)
           + 0.05 * (15 - temperatura)
           + rng.normal(0, 0.35, n)).round(2)

pd.DataFrame({"velocidad": velocidad, "peso": peso, "temperatura": temperatura, "consumo": consumo}) \
    .to_csv("consumo3.csv", index=False)
print("✅ Fichero 'consumo3.csv' creado")

In [ ]:
df = pd.read_csv("consumo3.csv")
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

for eje, col in zip(ejes, ["velocidad", "peso", "temperatura"]):
    eje.scatter(df[col], df["consumo"], alpha=0.5, s=14)
    eje.set_xlabel(col)
    eje.set_title(f"Consumo según {col}")

ejes[0].set_ylabel("Consumo (l/100 km)")
plt.tight_layout()
plt.show()

👀 La **velocidad** dibuja una "U" (relación **curva**). El **peso** y la **temperatura** tienen efectos más suaves.

## 7. Comparar tres modelos

Separamos entrenamiento y prueba (como siempre) y comparamos:

1. **Regresión lineal** (rectas)
2. **Un árbol de decisión solo, sin límites**
3. **Gradient Boosting**

Con dos medidas: **MAE** (error medio en litros) y **R²** (nota de 0 a 1).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df[["velocidad", "peso", "temperatura"]]
y_c = df["consumo"]

X_train, X_test, y_train, y_test = train_test_split(X, y_c, test_size=0.2, random_state=42)

modelos = {
    "Regresión lineal": LinearRegression(),
    "Un árbol (sin límites)": DecisionTreeRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

filas = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    filas.append({
        "modelo": nombre,
        "error medio MAE (litros)": round(mean_absolute_error(y_test, pred), 2),
        "R² prueba": round(r2_score(y_test, pred), 2),
        "R² entrenamiento": round(r2_score(y_train, modelo.predict(X_train)), 2),
    })

pd.DataFrame(filas)

👀 **Qué ha pasado:**
- La **regresión lineal** no sabe dibujar la "U" de la velocidad: se queda **muy atrás**.
- El **árbol solo** sin límites saca R² = 1 en entrenamiento (**memoriza**) pero **peor en la prueba**: sobreajuste.
- El **Gradient Boosting** es **el mejor en la prueba**: aprende la forma de la "U" **sin que le digamos nada** y sin memorizar.

> 🎯 Lo vimos con la regresión polinomial: había que **decidir nosotros** el grado. Aquí el modelo **descubre la forma solo**.

## 8. ¿Cuántos árboles? ¿Qué tasa de aprendizaje?

Dos "mandos" principales:

| Mando | Qué controla |
|---|---|
| `n_estimators` | **Cuántos árboles** (rondas) |
| `learning_rate` | **Cuánto pesa cada árbol** (tasa pequeña = aprende despacio pero mejor) |

### Número de árboles

In [ ]:
filas = []
for n_arboles in [1, 5, 20, 50, 100, 300, 600]:
    m = GradientBoostingRegressor(n_estimators=n_arboles, learning_rate=0.1, max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    filas.append({
        "árboles": n_arboles,
        "R² entrenamiento": round(r2_score(y_train, m.predict(X_train)), 3),
        "R² prueba": round(r2_score(y_test, m.predict(X_test)), 3),
    })

tabla_arboles = pd.DataFrame(filas).set_index("árboles")
tabla_arboles

In [ ]:
plt.plot(tabla_arboles.index, tabla_arboles["R² entrenamiento"], marker="o", color="seagreen", label="Entrenamiento")
plt.plot(tabla_arboles.index, tabla_arboles["R² prueba"], marker="o", color="darkorange", label="Prueba (examen)")
plt.xlabel("Número de árboles")
plt.ylabel("R²")
plt.title("Rendimiento según el número de árboles")
plt.legend()
plt.show()

👀 **La gráfica de siempre:**
- Con **muy pocos árboles** (1-5) el modelo es **demasiado simple**: R² bajo en los dos.
- Sube **rápido** y luego **se estabiliza**.
- A partir de cierto punto, el R² de **entrenamiento sigue subiendo** (se acerca a 1) mientras que el de **prueba se queda quieto o baja un poco**: el modelo empieza a **memorizar el ruido**.

> 📏 **Regla práctica:** no hace falta un número exagerado de árboles. Con 100-300 suele ir muy bien.

### Tasa de aprendizaje

In [ ]:
filas = []
for tasa in [1.0, 0.3, 0.1, 0.01]:
    m = GradientBoostingRegressor(n_estimators=100, learning_rate=tasa, max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    filas.append({
        "tasa de aprendizaje": tasa,
        "R² entrenamiento": round(r2_score(y_train, m.predict(X_train)), 3),
        "R² prueba": round(r2_score(y_test, m.predict(X_test)), 3),
    })

pd.DataFrame(filas)

👀 Con **100 árboles fijos**:
- **Tasa muy alta (1,0):** da pasos grandes y bruscos; **sobreajusta** (entrenamiento casi perfecto, prueba peor).
- **Tasa muy baja (0,01):** da pasos tan pequeños que con solo 100 árboles **no le da tiempo a llegar** (infraajuste).
- **Tasas intermedias (0,1-0,3):** el equilibrio.

> 🔗 **Van de la mano:** tasa baja necesita **más árboles**; tasa alta, **menos**. Lo habitual es una tasa pequeña (0,05-0,1) con bastantes árboles.

## 9. ¿Qué variables importan?

Un modelo de boosting puede decirnos **cuánto ha usado cada variable** (`feature_importances_`):

In [ ]:
mejor = GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)

importancias = pd.Series(mejor.feature_importances_, index=X.columns).sort_values()

importancias.plot(kind="barh", color="steelblue", figsize=(6, 3))
plt.title("Importancia de cada variable")
plt.xlabel("Importancia (la suma de las tres es 1)")
plt.show()

importancias.sort_values(ascending=False).round(2)

👀 La **velocidad** es, con diferencia, la variable más importante (es la que dibuja la "U"). La temperatura va después y el peso es la menos importante. **Coincide con cómo se fabricaron los datos.**

### Usar el modelo

In [ ]:
viaje = pd.DataFrame({"velocidad": [90], "peso": [1450], "temperatura": [5.0]})
print(f"Consumo estimado: {mejor.predict(viaje)[0]:.2f} litros cada 100 km")
print(f"(el modelo se equivoca de media en unos {mean_absolute_error(y_test, mejor.predict(X_test)):.2f} litros)")

## 10. Resumen

### La receta del Gradient Boosting

| Paso | Código |
|---|---|
| Modelo (para predecir números) | `GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)` |
| Modelo (para clasificar) | `GradientBoostingClassifier(...)` |
| Entrenar y predecir | `.fit(X_train, y_train)` y `.predict(X_test)` |
| Ver la importancia de variables | `modelo.feature_importances_` |

### Ideas clave

1. **Boosting = muchos árboles pequeños, uno tras otro, cada uno corrigiendo los errores del anterior.**
2. La predicción final es la **suma** de todos (cada uno con un peso pequeño).
3. **Capta curvas y formas complicadas por sí solo**, sin decidir nosotros nada.
4. Dos mandos principales: **número de árboles** y **tasa de aprendizaje** (van de la mano).
5. **Puede sobreajustar** si se pasa de árboles o de tasa: hay que **mirar la prueba**.
6. Es **menos fácil de explicar** que una regresión lineal.

### 📖 Mini glosario

| Palabra | Significado sencillo |
|---|---|
| **Boosting** | Combinar modelos pequeños uno tras otro, cada uno corrigiendo al anterior |
| **Residuo** | El error: lo que falta entre la predicción y el valor real |
| **Tasa de aprendizaje** | Cuánto pesa cada árbol nuevo (pequeña = prudente) |
| **Ronda / estimador** | Cada árbol que se añade |

## ✍️ Ejercicios

**Nivel 1 – Para empezar**
1. En la Parte A, cambia `max_depth=2` por `1` en el bucle a mano. ¿Cómo cambia la línea roja tras 50 rondas? (Con profundidad 1 cada árbol solo hace **una** pregunta.)
2. ¿Por qué el R² de **entrenamiento** sube siempre al añadir árboles, y el de **prueba** no?
3. Explica con tus palabras qué es un **residuo**.

**Nivel 2 – Practicar**
4. En la Parte A, usa una tasa de aprendizaje de `1.0`. ¿Qué pasa con los errores a las 5 rondas? ¿Y a las 50?
5. En la Parte B, entrena un boosting con `max_depth=1` y otro con `max_depth=6`. ¿Cuál rinde mejor en la prueba?
6. Quita la variable `peso` y vuelve a entrenar. ¿Cambia mucho el R²?

**Nivel 3 – Reto**
7. Compara el boosting con un **`RandomForestRegressor`** (cambia solo el nombre del modelo). ¿Cuál gana en este problema?

---
### 🎯 ¿Y ahora qué?
Después del boosting veremos sus "hermanos": los **árboles de decisión** y el **Random Forest**, que combina árboles de otra manera (muchos a la vez, no uno tras otro).